# Assignment-6: Weather Condition Classification using SVM and Open-Meteo API

**Objective:** Classify weather as **Cool** or **Warm** using meteorological data from Open-Meteo API with Support Vector Machine (SVM).

- **API Documentation:** [Open-Meteo](https://open-meteo.com/)
- **Libraries Used:** `requests`, `pandas`, `numpy`, `scikit-learn`, `matplotlib`, `seaborn`
- **Location:** New Delhi, India (lat: 28.6139, lon: 77.2090)
- **Target Variable:** `Weather_Class` — Warm (≥25°C) vs Cool (<25°C)

In [ ]:
# ============================================================
# Import Required Libraries
# ============================================================
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

import warnings
warnings.filterwarnings('ignore')

print('All libraries imported successfully!')

## Task 1: Data Collection and Understanding (2 Marks)

Fetch weather data from Open-Meteo API for New Delhi and convert JSON to Pandas DataFrame.

In [ ]:
# ============================================================
# 1. Fetch Weather Data from Open-Meteo API
# ============================================================

# API endpoint for New Delhi
url = (
    "https://api.open-meteo.com/v1/forecast"
    "?latitude=28.6139&longitude=77.2090"
    "&hourly=temperature_2m,relative_humidity_2m,surface_pressure,wind_speed_10m"
    "&forecast_days=7"
)

response = requests.get(url)
data = response.json()

print(f"Status Code: {response.status_code}")
print(f"Keys in response: {list(data.keys())}")

In [ ]:
# ============================================================
# 2. Convert JSON to Pandas DataFrame
# ============================================================

hourly = data['hourly']

df = pd.DataFrame({
    'time': hourly['time'],
    'temperature_2m': hourly['temperature_2m'],
    'relative_humidity_2m': hourly['relative_humidity_2m'],
    'surface_pressure': hourly['surface_pressure'],
    'wind_speed_10m': hourly['wind_speed_10m']
})

# Convert time to datetime
df['time'] = pd.to_datetime(df['time'])

print(f"Shape of DataFrame: {df.shape}")
print(f"\nColumns: {list(df.columns)}")

In [ ]:
# ============================================================
# 3. Display First Five Records
# ============================================================
df.head()

In [ ]:
# ============================================================
# 4. Identify Input Features and Target Variable
# ============================================================

# Create Target Variable: Weather_Class
# Warm → Temperature ≥ 25°C
# Cool → Temperature < 25°C

df['Weather_Class'] = df['temperature_2m'].apply(
    lambda x: 'Warm' if x >= 25 else 'Cool'
)

print("===== FEATURE IDENTIFICATION =====")
print("\nInput Features (X):")
print("  1. temperature_2m        → Temperature at 2m (°C)")
print("  2. relative_humidity_2m  → Relative Humidity at 2m (%)")
print("  3. surface_pressure      → Surface Pressure (hPa)")
print("  4. wind_speed_10m        → Wind Speed at 10m (km/h)")
print("\nTarget Variable (y):")
print("  Weather_Class            → Binary: 'Warm' or 'Cool'")
print("\nClass Distribution:")
print(df['Weather_Class'].value_counts())

## Task 2: Data Preprocessing (2 Marks)

Steps: Check missing values → Remove unnecessary columns → Encode target → Split data → Standardize features.

In [ ]:
# ============================================================
# 1. Check for Missing Values
# ============================================================
print("Missing Values per Column:")
print(df.isnull().sum())
print(f"\nTotal Missing: {df.isnull().sum().sum()}")

In [ ]:
# ============================================================
# 2. Remove Unnecessary Columns
# ============================================================

# Drop 'time' as it is not a predictive feature
df_processed = df.drop(columns=['time']).copy()

print("Columns after removal:")
print(list(df_processed.columns))
print(f"\nShape: {df_processed.shape}")

In [ ]:
# ============================================================
# 3. Encode the Target Variable
# ============================================================

label_encoder = LabelEncoder()
df_processed['Weather_Class_Encoded'] = label_encoder.fit_transform(
    df_processed['Weather_Class']
)

print("Label Encoding Mapping:")
for i, cls in enumerate(label_encoder.classes_):
    print(f"  {cls} → {i}")

# Note: 'Cool' = 0, 'Warm' = 1 (alphabetical order)
print("\nEncoded Distribution:")
print(df_processed['Weather_Class_Encoded'].value_counts())

In [ ]:
# ============================================================
# 4. Split Dataset: 80% Training | 20% Testing
# ============================================================

# Define X (features) and y (target)
X = df_processed[['temperature_2m', 'relative_humidity_2m',
                  'surface_pressure', 'wind_speed_10m']]
y = df_processed['Weather_Class_Encoded']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Training set size: {X_train.shape[0]} samples")
print(f"Testing set size : {X_test.shape[0]} samples")
print(f"\nTraining class distribution:\n{y_train.value_counts()}")
print(f"\nTesting class distribution:\n{y_test.value_counts()}")

In [ ]:
# ============================================================
# 5. Standardize Feature Values using StandardScaler
# ============================================================

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert back to DataFrame for readability
X_train_scaled_df = pd.DataFrame(
    X_train_scaled, columns=X.columns
)
X_test_scaled_df = pd.DataFrame(
    X_test_scaled, columns=X.columns
)

print("Training Features After Standardization (first 5 rows):")
print(X_train_scaled_df.head())
print("\nMean (should be ~0):")
print(np.round(X_train_scaled_df.mean(), 4))
print("\nStd Dev (should be ~1):")
print(np.round(X_train_scaled_df.std(), 4))

## Task 3: Model Development (3 Marks)

Build and train an SVM Classifier with **RBF kernel**.

In [ ]:
# ============================================================
# Build SVM Classifier with RBF Kernel
# ============================================================

svm_model = SVC(
    kernel='rbf',      # Radial Basis Function kernel
    C=1.0,             # Regularization parameter
    gamma='scale',     # Kernel coefficient
    random_state=42
)

# Train the model
svm_model.fit(X_train_scaled, y_train)

print("SVM Model trained successfully!")
print(f"Kernel: {svm_model.kernel}")
print(f"C: {svm_model.C}")
print(f"Gamma: {svm_model.gamma}")

In [ ]:
# ============================================================
# Predict on Test Set
# ============================================================

y_pred = svm_model.predict(X_test_scaled)

# Create prediction comparison DataFrame
pred_df = pd.DataFrame({
    'Actual': y_test.values,
    'Predicted': y_pred
})

# Map back to class names
pred_df['Actual_Class'] = pred_df['Actual'].map({0: 'Cool', 1: 'Warm'})
pred_df['Predicted_Class'] = pred_df['Predicted'].map({0: 'Cool', 1: 'Warm'})

print("First 10 Predictions:")
print(pred_df[['Actual_Class', 'Predicted_Class']].head(10))

## Task 4: Model Evaluation (2 Marks)

Evaluate using Accuracy, Precision, Recall, F1-Score, and Confusion Matrix.

In [ ]:
# ============================================================
# Calculate Evaluation Metrics
# ============================================================

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='binary', pos_label=1)
recall = recall_score(y_test, y_pred, average='binary', pos_label=1)
f1 = f1_score(y_test, y_pred, average='binary', pos_label=1)

print("===== MODEL EVALUATION METRICS =====")
print(f"Accuracy  : {accuracy:.4f}  ({accuracy*100:.2f}%)")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1-Score  : {f1:.4f}")

print("\n===== DETAILED CLASSIFICATION REPORT =====")
print(classification_report(
    y_test, y_pred,
    target_names=['Cool', 'Warm'],
    digits=4
))

In [ ]:
# ============================================================
# Confusion Matrix
# ============================================================

cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=['Cool', 'Warm'],
    yticklabels=['Cool', 'Warm']
)
plt.title('Confusion Matrix - SVM (RBF Kernel)', fontsize=14, fontweight='bold')
plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('True Label', fontsize=12)
plt.tight_layout()
plt.show()

print("Confusion Matrix Values:")
print(f"  True Negatives  (Cool → Cool) : {cm[0,0]}")
print(f"  False Positives (Cool → Warm) : {cm[0,1]}")
print(f"  False Negatives (Warm → Cool) : {cm[1,0]}")
print(f"  True Positives  (Warm → Warm) : {cm[1,1]}")

In [ ]:
# ============================================================
# 3 Observations Based on Model Performance
# ============================================================

print("===== 3 OBSERVATIONS =====")
print()
print("1. HIGH CLASSIFICATION ACCURACY:")
print(f"   The SVM model with RBF kernel achieves {accuracy*100:.2f}% accuracy,")
print("   indicating strong separability between Cool and Warm weather")
print("   conditions using the selected meteorological features.")
print()
print("2. TEMPERATURE IS THE DOMINANT FEATURE:")
print("   Since the target (Weather_Class) is directly derived from temperature,")
print("   the model leverages this strong correlation. However, humidity,")
print("   pressure, and wind speed provide additional contextual boundaries.")
print()
print("3. FEATURE SCALING IS ESSENTIAL FOR SVM:")
print("   Features like surface_pressure (~1000 hPa) and wind_speed (~5 km/h)")
print("   have vastly different scales. StandardScaler ensures the RBF kernel")
print("   computes meaningful distance metrics without bias toward larger-scale features.")

## Task 5: Conclusion (1 Mark)

Write a 100–150 word conclusion covering key findings, importance of feature scaling, and one advantage + one limitation of SVM.

In [ ]:
conclusion = """
===== CONCLUSION =====

This assignment successfully demonstrates weather condition classification using
Support Vector Machine with the Open-Meteo API. The SVM classifier with an RBF
kernel achieved high accuracy in distinguishing between Cool (<25°C) and Warm
(≥25°C) weather conditions using temperature, relative humidity, surface pressure,
and wind speed as input features.

Feature scaling proved critical for SVM performance because the algorithm relies
on distance-based optimization. Without standardization, features like surface
pressure (in hundreds) would dominate over wind speed (in single digits), leading
to biased hyperplane placement and poor generalization.

One key advantage of SVM is its effectiveness in high-dimensional spaces and its
ability to model complex, non-linear decision boundaries through kernel tricks.
However, a significant limitation is its computational cost on large datasets, as
training time scales quadratically with the number of samples, making it less
suitable for massive real-time weather streaming applications.
"""

print(conclusion)
print(f"\nWord Count: {len(conclusion.split())} words")

---

## Appendix: Data Summary & Visualization

In [ ]:
# ============================================================
# Feature Distribution by Weather Class
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
features = ['temperature_2m', 'relative_humidity_2m',
            'surface_pressure', 'wind_speed_10m']
titles = ['Temperature (°C)', 'Relative Humidity (%)',
          'Surface Pressure (hPa)', 'Wind Speed (km/h)']

for ax, feat, title in zip(axes.flatten(), features, titles):
    for cls, color in zip(['Cool', 'Warm'], ['#3498db', '#e74c3c']):
        subset = df[df['Weather_Class'] == cls][feat]
        ax.hist(subset, bins=15, alpha=0.6, label=cls, color=color, edgecolor='black')
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel(title)
    ax.set_ylabel('Frequency')
    ax.legend()

plt.suptitle('Feature Distributions by Weather Class', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Data Summary Statistics
# ============================================================
print("===== DATA SUMMARY STATISTICS =====")
print(df[features].describe().round(2))